<a href="https://colab.research.google.com/github/Zhengdao-Lu/roblem-Set-3-Econ_3916_PS3.ipynb/blob/main/ps3_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [2]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [5]:
N_CUSTOMERS = 4_000

N_WEST = 2_800
regions = np.array(["West"] * N_WEST + ["East"] * (N_CUSTOMERS - N_WEST))
rng.shuffle(regions)
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, N_CUSTOMERS)

customers_df = pd.DataFrame({
    "customer_id": np.arange(1, N_CUSTOMERS + 1),
    "region": regions,
    "revenue": np.maximum(
        REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, N_CUSTOMERS),
        REV_FLOOR,
    ).round(2),
    "signup_date": pd.Timestamp("2022-01-01")
                   + pd.to_timedelta(rng.integers(0, 1095, N_CUSTOMERS), unit="D"),
})
orders_df = pd.DataFrame({
    "customer_id": np.repeat(customers_df["customer_id"], n_orders),
})
orders_df.insert(0, "order_id", np.arange(1, len(orders_df) + 1))

con.register("customers_df", customers_df)
con.register("orders_df", orders_df)
con.execute("CREATE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE TABLE orders AS SELECT * FROM orders_df")
print(con.sql("SELECT COUNT(*) AS customers FROM customers").df())
print(con.sql("SELECT COUNT(*) AS west FROM customers WHERE region = 'West'").df())
print(con.sql("SELECT COUNT(*) AS orders FROM orders").df())

   customers
0       4000
   west
0  2800
   orders
0   27211


### Step 1.2: Write both queries (7 pts)

In [6]:
low_sql = """
    SELECT COUNT(*) AS rows_used, ROUND(AVG(revenue), 2) AS average_revenue
    FROM customers
    WHERE region = 'West'
"""

high_sql = """
    SELECT COUNT(*) AS rows_used, ROUND(AVG(c.revenue), 2) AS average_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

low = con.sql(low_sql).df().iloc[0]
high = con.sql(high_sql).df().iloc[0]
ratio = high["average_revenue"] / low["average_revenue"]
print(f"Per customer: ${low['average_revenue']:,.2f} over {int(low['rows_used']):,} rows")
print(f"After join:  ${high['average_revenue']:,.2f} over {int(high['rows_used']):,} rows")
print(f"Ratio: {ratio:.3f}")

assert 780 <= low["average_revenue"] <= 845
assert 1160 <= high["average_revenue"] <= 1320
assert 1.45 <= ratio <= 1.60

Per customer: $811.17 over 2,800 rows
After join:  $1,231.82 over 19,082 rows
Ratio: 1.519


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

*Your answer here.*

Finance's lower number is the right one for average revenue per customer. In this synthetic warehouses the unjoined query giver 811.17 over 2800 West customers because each customer has exactly one vote. THe joined query giver $1231.82 over 19082 order rowas. This is artithmaticly valid, but its unit does not match the lable.

### Step 1.3: The rule (10 pts)

In [7]:
def check_join(join_sql):
    before = con.sql("SELECT COUNT(*) FROM customers WHERE region = 'West'").fetchall()[0][0]
    after = con.sql(f"SELECT COUNT(*) FROM ({join_sql}) AS joined_rows").fetchall()[0][0]
    if after != before:
        print(f"Warning: {before:,} rows before the join, {after:,} after")

bad_join = """
    SELECT c.customer_id, c.revenue
    FROM customers c JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

good_join = """
    SELECT c.customer_id, c.revenue
    FROM customers c
    LEFT JOIN (SELECT customer_id, COUNT(*) AS n_orders
               FROM orders GROUP BY customer_id) o
    ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

print("Bad join:")
check_join(bad_join)
print("Good join:")
check_join(good_join)

Bad join:
Good join:


*Your answer here.*



## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [8]:
missing_ids = rng.choice(customers_df["customer_id"], size=600, replace=False)
con.register("missing_ids", pd.DataFrame({"customer_id": missing_ids}))
con.execute("""
    UPDATE customers SET region = NULL
    WHERE customer_id IN (SELECT customer_id FROM missing_ids)
""")
missing = con.sql("SELECT COUNT(*) FROM customers WHERE region IS NULL").fetchall()[0][0]
print(f"Missing region: {missing:,} of {N_CUSTOMERS:,} customers")

Missing region: 600 of 4,000 customers


### Step 2.2: Lose them, then find them (15 pts)

In [9]:
lost_query = "SELECT COUNT(*) FROM customers WHERE region != 'West'"
kept_query = "SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'"
lost_count = con.sql(lost_query).fetchall()[0][0]
kept_count = con.sql(kept_query).fetchall()[0][0]
print(f"region != 'West': {lost_count:,} rows")
print(f"Including unknown regions: {kept_count:,} rows")
print(f"Difference: {kept_count - lost_count:,} rows")

region != 'West': 1,046 rows
Including unknown regions: 1,646 rows
Difference: 600 rows


*Your answer here.*

region != 'West' retuned 1046 rows, region IS DISTINCT FROM 'West' retuned 1646. region != 'West' evaluate to unknown for NULL regions, and WHERE retains only true conditoons. If mission-region customers are systematically newer, smaller, or more likely to chur, the first query selects a different population and can make ovserved regional revenue look hgher and chrum look lower than they really are.

### Step 2.3: Aggregate without collapsing (10 pts)

In [10]:
window_sql = """
    SELECT customer_id, region, revenue,
           ROUND(AVG(revenue) OVER (PARTITION BY region), 2) AS region_avg,
           ROUND(revenue - AVG(revenue) OVER (PARTITION BY region), 2) AS vs_region
    FROM customers
    ORDER BY customer_id
"""
with_average = con.sql(window_sql).df()
print(with_average.head(8).to_string(index=False))
print(f"Rows before: {N_CUSTOMERS:,}; rows after: {len(with_average):,}")
assert len(with_average) == N_CUSTOMERS

 customer_id region  revenue  region_avg  vs_region
           1   West   760.63      810.33     -49.70
           2   West   736.81      810.33     -73.52
           3   West   713.32      810.33     -97.01
           4   None   525.75      813.04    -287.29
           5   West   837.24      810.33      26.91
           6   West  1260.78      810.33     450.45
           7   West   879.82      810.33      69.49
           8   East   385.74      804.97    -419.23
Rows before: 4,000; rows after: 4,000


## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [11]:
causal_rng = np.random.default_rng(2026)
n = 20_000
size = np.maximum(causal_rng.normal(100, 20, n), 10)
chance_west = 1 / (1 + np.exp(-(size - 100) / 15))
west = causal_rng.binomial(1, chance_west, n)
TRUE_EFFECT = 25
revenue = 400 + 8 * size + TRUE_EFFECT * west + causal_rng.normal(0, 100, n)

programme_df = pd.DataFrame({"west": west, "size": size, "revenue": revenue})
naive = programme_df.loc[west == 1, "revenue"].mean() - programme_df.loc[west == 0, "revenue"].mean()
adjusted = smf.ols("revenue ~ west + size", data=programme_df).fit().params["west"]
print(f"Truth: ${TRUE_EFFECT:.2f}")
print(f"Naive West–East gap: ${naive:.2f}")
print(f"Adjusting for existing size: ${adjusted:.2f}")

Truth: $25.00
Naive West–East gap: $185.59
Adjusting for existing size: $24.06


*Your answer here.*

In the simulation, the true program effect is 25. The vaice West-East gap is 185.59 because West customers already had larger accounts, which independently raise renvuen. In the actual warehouse, the difference alone cannot establish causation, observational adjustment is credible only if the important common causes are measured and modeled approprately.

### Step 3.2: The control that ruins it (13 pts)

In [12]:
average_tickets = np.maximum(0.2, 8 + 3 * west + 0.015 * (revenue - 1200))
programme_df["tickets"] = causal_rng.poisson(average_tickets)
with_tickets = smf.ols("revenue ~ west + size + tickets", data=programme_df).fit().params["west"]
print(f"Truth: ${TRUE_EFFECT:.2f}")
print(f"Naive gap: ${naive:.2f}")
print(f"Adjusting for size: ${adjusted:.2f}")
print(f"Also adjusting for tickets: ${with_tickets:.2f}")

Truth: $25.00
Naive gap: $185.59
Adjusting for size: $24.06
Also adjusting for tickets: $-18.31


*Your answer here.*

West customers already had larger accounts before the programme, their high revenue therefore cannnot be credited to account management. In a simulation where the tru grogramme effect is 25, acounting for existing size moves the estimate from the raw 185.59 gap to 24.06.Comparing customer with the same ticket count fistorts the comparison and moves the estimate to -18.31, farther from the truth. For the real rollout, I would use pre-programme data and comparison design, to randomized assignment before presenting the causal claim.




## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [23]:
def review_query(con, sql):
    """The model's function from the P.R.I.M.E. prompt, then your corrections:
    returns a list of structured warnings for one SELECT."""
    ...

In [22]:
# Run review_query on your Phase 1 and Phase 2 queries.

#Phase 1 query

"""Small DuckDB reviewer for ordinary single-SELECT customers/orders queries.

Returns warning dictionaries with code, severity, message, and details. Complex
SQL that cannot be inspected safely receives UNSUPPORTED_QUERY, never a clean bill.
"""

import re


_IDENT = r"[A-Za-z_][A-Za-z_0-9]*"
_TABLE = re.compile(rf"^({_IDENT})(?:\s+(?:AS\s+)?({_IDENT}))?$", re.I)
_JOIN = re.compile(r"\b(?:(LEFT|RIGHT|FULL|INNER|CROSS)\s+(?:OUTER\s+)?)?JOIN\b", re.I)
_CLAUSES = ("FROM", "WHERE", "GROUP BY", "HAVING", "ORDER BY", "LIMIT", "QUALIFY")
_AGGREGATE = re.compile(r"\b(SUM|AVG|COUNT)\s*\(\s*(?!DISTINCT\b)", re.I)


def _top_level_clauses(sql):
    """Find clause boundaries while ignoring quoted text and parentheses."""
    result = {}
    depth = 0
    quote = None
    i = 0
    while i < len(sql):
        char = sql[i]
        if quote:
            if char == quote:
                if i + 1 < len(sql) and sql[i + 1] == quote:
                    i += 2
                    continue
                quote = None
            i += 1
            continue
        if char in ("'", '"'):
            quote = char
        elif char == "(":
            depth += 1
        elif char == ")":
            depth -= 1
        elif depth == 0:
            for name in _CLAUSES:
                if (sql[i:i + len(name)].upper() == name
                        and (i == 0 or not (sql[i - 1].isalnum() or sql[i - 1] == "_"))
                        and (i + len(name) == len(sql) or not (sql[i + len(name)].isalnum() or sql[i + len(name)] == "_"))):
                    result.setdefault(name, (i, i + len(name)))
                    i += len(name) - 1
                    break
        i += 1
    return result


def _parts(sql):
    positions = _top_level_clauses(sql)
    if "FROM" not in positions:
        raise ValueError("SELECT must have a FROM clause")
    ordered = sorted(positions.items(), key=lambda item: item[1][0])
    chunks = {}
    for n, (name, (_, end)) in enumerate(ordered):
        next_start = ordered[n + 1][1][0] if n + 1 < len(ordered) else len(sql)
        chunks[name] = sql[end:next_start].strip()
    chunks["SELECT"] = sql[len("SELECT"):positions["FROM"][0]].strip()
    return chunks


def _table_spec(text):
    match = _TABLE.fullmatch(text.strip())
    if not match:
        raise ValueError("Only plain table names and optional aliases are supported")
    table, alias = match.groups()
    return table, alias or table


def _from_spec(text):
    match = _JOIN.search(text)
    if not match:
        table, alias = _table_spec(text)
        return table, alias, None, {alias: table}, f"__review_base {alias}"
    left = text[:match.start()].strip()
    right = text[match.end():].strip()
    on = re.search(r"\bON\b", right, re.I)
    if not on or _JOIN.search(right):
        raise ValueError("Only one JOIN with an ON clause is supported")
    base_table, base_alias = _table_spec(left)
    other_table, other_alias = _table_spec(right[:on.start()])
    condition = right[on.end():].strip()
    if not re.fullmatch(rf"\s*{_IDENT}\.{_IDENT}\s*=\s*{_IDENT}\.{_IDENT}\s*", condition):
        raise ValueError("Only a single equality JOIN condition is supported")
    kind = (match.group(1) or "INNER").upper()
    aliases = {base_alias: base_table, other_alias: other_table}
    joined_from = f"__review_base {base_alias} {kind} JOIN {other_table} {other_alias} ON {condition}"
    return base_table, base_alias, kind, aliases, joined_from


def _nullable_columns(con, aliases, predicate):
    refs = re.findall(rf"\b({_IDENT})\.({_IDENT})\b", predicate)
    for alias, table in aliases.items():
        columns = {row[1].lower(): row[3] for row in con.execute(
            "SELECT * FROM pragma_table_info(?)", [table]).fetchall()}
        for column, not_null in columns.items():
            if not not_null and re.search(rf"(?<![\w.]){re.escape(column)}\b", predicate, re.I):
                refs.append((alias, column))
    unique = []
    for alias, column in refs:
        if alias not in aliases:
            continue
        table = aliases[alias]
        metadata = {row[1].lower(): row[3] for row in con.execute(
            "SELECT * FROM pragma_table_info(?)", [table]).fetchall()}
        if not metadata.get(column.lower(), True) and (alias, column) not in unique:
            unique.append((alias, column))
    return unique


def review_query(con, sql):
    """Review a single SELECT; return structured warnings, without printing.

    The diagnostic query counts joined rows and DISTINCT base row IDs in one
    scan of the join. Their difference exposes fan-out even when dropped rows
    make the *net* row-count change zero. Counting the base table separately
    does not rerun the join. NULL-filter counts share that same joined scan.

    SQL WHERE retains only TRUE. A comparison with NULL yields UNKNOWN, so
    rows can disappear and change a dashboard metric. This is a correctness
    issue, not a formatting or SQL-style preference.
    """
    if not isinstance(sql, str) or not re.match(r"^\s*SELECT\b", sql, re.I):
        raise ValueError("review_query accepts one SELECT statement")
    sql = sql.strip().rstrip(";").strip()
    if ";" in sql:
        raise ValueError("review_query accepts one SELECT statement")
    con.execute("EXPLAIN " + sql)  # Ask DuckDB to validate syntax and bindings.
    try:
        chunks = _parts(sql)
        base_table, base_alias, join_kind, aliases, joined_from = _from_spec(chunks["FROM"])
    except ValueError as exc:
        return [{"code": "UNSUPPORTED_QUERY", "severity": "warning",
                 "message": str(exc), "details": {}}]

    predicate = chunks.get("WHERE", "")
    nullable = _nullable_columns(con, aliases, predicate) if predicate else []
    # Explicit NULL handling is intentional; it does not silently discard UNKNOWN.
    nullable = [(a, c) for a, c in nullable if not re.search(
        rf"\b(?:{re.escape(a)}\.)?{re.escape(c)}\s+IS\s+(?:NOT\s+)?NULL\b"
        rf"|\bCOALESCE\s*\(\s*(?:{re.escape(a)}\.)?{re.escape(c)}\b",
        predicate, re.I)]

    filter_sql = f" FILTER (WHERE ({predicate}))" if predicate else ""
    select_stats = [f"COUNT(*){filter_sql} AS joined_rows",
                    f"COUNT(DISTINCT {base_alias}.__review_rowid){filter_sql} AS matched_base_rows"]
    for index, (alias, column) in enumerate(nullable):
        select_stats.append(
            f"COUNT(*) FILTER (WHERE {alias}.{column} IS NULL AND ({predicate}) IS NOT TRUE) AS null_{index}")
    # The join appears once in this diagnostic statement; no full-result fetch.
    diagnostic = (f"WITH __review_base AS (SELECT *, ROW_NUMBER() OVER () AS __review_rowid "
                  f"FROM {base_table}) SELECT {', '.join(select_stats)} FROM {joined_from}")
    values = con.execute(diagnostic).fetchone()
    joined_rows, matched_base_rows = values[:2]
    # Apply a base-only WHERE to the baseline so the comparison uses the same
    # population (for example, West customers), not every customer in the table.
    other_aliases = [a for a in aliases if a != base_alias]
    base_predicate = predicate if predicate and not any(re.search(
        rf"\b{re.escape(a)}\.", predicate, re.I) for a in other_aliases) else ""
    try:
        base_rows = con.execute(
            f"SELECT COUNT(*) FROM {base_table} {base_alias}"
            + (f" WHERE {base_predicate}" if base_predicate else "")).fetchone()[0]
    except Exception:
        # An unqualified WHERE column can belong to the joined table.
        base_rows = con.execute(f"SELECT COUNT(*) FROM {base_table}").fetchone()[0]
    fanned_out_rows = joined_rows - matched_base_rows
    warnings = []
    if join_kind and (joined_rows != base_rows or fanned_out_rows > 0):
        warnings.append({"code": "JOIN_ROW_COUNT_CHANGE", "severity": "warning",
                         "message": "Join changes row multiplicity relative to the left table.",
                         "details": {"base_rows": base_rows, "joined_rows": joined_rows,
                                     "matched_base_rows": matched_base_rows,
                                     "fanned_out_rows": fanned_out_rows}})
    for index, (alias, column) in enumerate(nullable):
        warnings.append({"code": "NULL_FILTER", "severity": "warning",
                         "message": "WHERE can discard UNKNOWN for a nullable column.",
                         "details": {"column": f"{alias}.{column}",
                                     "dropped_null_rows": values[index + 2]}})
    if join_kind and fanned_out_rows > 0 and _AGGREGATE.search(chunks["SELECT"]):
        warnings.append({"code": "FANOUT_AGGREGATE", "severity": "warning",
                         "message": "Aggregate is computed after a fan-out; verify the intended grain.",
                         "details": {"fanned_out_rows": fanned_out_rows}})
    return warnings

In [21]:
#Phase 2
"""Small DuckDB reviewer for ordinary single-SELECT customers/orders queries.

Returns warning dictionaries with code, severity, message, and details. Complex
SQL that cannot be inspected safely receives UNSUPPORTED_QUERY, never a clean bill.
"""

import re


_IDENT = r"[A-Za-z_][A-Za-z_0-9]*"
_TABLE = re.compile(rf"^({_IDENT})(?:\s+(?:AS\s+)?({_IDENT}))?$", re.I)
_JOIN = re.compile(r"\b(?:(LEFT|RIGHT|FULL|INNER|CROSS)\s+(?:OUTER\s+)?)?JOIN\b", re.I)
_CLAUSES = ("FROM", "WHERE", "GROUP BY", "HAVING", "ORDER BY", "LIMIT", "QUALIFY")
_AGGREGATE = re.compile(r"\b(SUM|AVG|COUNT)\s*\(\s*(?!DISTINCT\b)", re.I)


def _top_level_clauses(sql):
    """Find clause boundaries while ignoring quoted text and parentheses."""
    result = {}
    depth = 0
    quote = None
    i = 0
    while i < len(sql):
        char = sql[i]
        if quote:
            if char == quote:
                if i + 1 < len(sql) and sql[i + 1] == quote:
                    i += 2
                    continue
                quote = None
            i += 1
            continue
        if char in ("'", '"'):
            quote = char
        elif char == "(":
            depth += 1
        elif char == ")":
            depth -= 1
        elif depth == 0:
            for name in _CLAUSES:
                if (sql[i:i + len(name)].upper() == name
                        and (i == 0 or not (sql[i - 1].isalnum() or sql[i - 1] == "_"))
                        and (i + len(name) == len(sql) or not (sql[i + len(name)].isalnum() or sql[i + len(name)] == "_"))):
                    result.setdefault(name, (i, i + len(name)))
                    i += len(name) - 1
                    break
        i += 1
    return result


def _parts(sql):
    positions = _top_level_clauses(sql)
    if "FROM" not in positions:
        raise ValueError("SELECT must have a FROM clause")
    ordered = sorted(positions.items(), key=lambda item: item[1][0])
    chunks = {}
    for n, (name, (_, end)) in enumerate(ordered):
        next_start = ordered[n + 1][1][0] if n + 1 < len(ordered) else len(sql)
        chunks[name] = sql[end:next_start].strip()
    chunks["SELECT"] = sql[len("SELECT"):positions["FROM"][0]].strip()
    return chunks


def _table_spec(text):
    match = _TABLE.fullmatch(text.strip())
    if not match:
        raise ValueError("Only plain table names and optional aliases are supported")
    table, alias = match.groups()
    return table, alias or table


def _from_spec(text):
    match = _JOIN.search(text)
    if not match:
        table, alias = _table_spec(text)
        return table, alias, None, {alias: table}, f"__review_base {alias}"
    left = text[:match.start()].strip()
    right = text[match.end():].strip()
    on = re.search(r"\bON\b", right, re.I)
    if not on or _JOIN.search(right):
        raise ValueError("Only one JOIN with an ON clause is supported")
    base_table, base_alias = _table_spec(left)
    other_table, other_alias = _table_spec(right[:on.start()])
    condition = right[on.end():].strip()
    if not re.fullmatch(rf"\s*{_IDENT}\.{_IDENT}\s*=\s*{_IDENT}\.{_IDENT}\s*", condition):
        raise ValueError("Only a single equality JOIN condition is supported")
    kind = (match.group(1) or "INNER").upper()
    aliases = {base_alias: base_table, other_alias: other_table}
    joined_from = f"__review_base {base_alias} {kind} JOIN {other_table} {other_alias} ON {condition}"
    return base_table, base_alias, kind, aliases, joined_from


def _nullable_columns(con, aliases, predicate):
    refs = re.findall(rf"\b({_IDENT})\.({_IDENT})\b", predicate)
    for alias, table in aliases.items():
        columns = {row[1].lower(): row[3] for row in con.execute(
            "SELECT * FROM pragma_table_info(?)", [table]).fetchall()}
        for column, not_null in columns.items():
            if not not_null and re.search(rf"(?<![\w.]){re.escape(column)}\b", predicate, re.I):
                refs.append((alias, column))
    unique = []
    for alias, column in refs:
        if alias not in aliases:
            continue
        table = aliases[alias]
        metadata = {row[1].lower(): row[3] for row in con.execute(
            "SELECT * FROM pragma_table_info(?)", [table]).fetchall()}
        if not metadata.get(column.lower(), True) and (alias, column) not in unique:
            unique.append((alias, column))
    return unique


def _whole_predicate_handles_null(predicate, alias, column):
    """Recognize a simple, explicitly NULL-aware predicate on this column."""
    ref = rf"(?:{re.escape(alias)}\.)?{re.escape(column)}"
    value = rf"(?:'(?:''|[^'])*'|\"(?:\"\"|[^\"])*\"|[+-]?\d+(?:\.\d+)?|{_IDENT}(?:\.{_IDENT})?)"
    return bool(re.fullmatch(
        rf"\s*{ref}\s+IS\s+(?:NOT\s+)?(?:NULL|DISTINCT\s+FROM\s+{value})\s*",
        predicate, re.I | re.S))


def review_query(con, sql):
    """Review a single SELECT; return structured warnings, without printing.

    The diagnostic query counts joined rows and DISTINCT base row IDs in one
    scan of the join. Their difference exposes fan-out even when dropped rows
    make the *net* row-count change zero. Counting the base table separately
    does not rerun the join. NULL-filter counts share that same joined scan.

    SQL WHERE retains only TRUE. A comparison with NULL yields UNKNOWN, so
    rows can disappear and change a dashboard metric. This is a correctness
    issue, not a formatting or SQL-style preference.
    """
    if not isinstance(sql, str) or not re.match(r"^\s*SELECT\b", sql, re.I):
        raise ValueError("review_query accepts one SELECT statement")
    sql = sql.strip().rstrip(";").strip()
    if ";" in sql:
        raise ValueError("review_query accepts one SELECT statement")
    con.execute("EXPLAIN " + sql)  # Ask DuckDB to validate syntax and bindings.
    try:
        chunks = _parts(sql)
        base_table, base_alias, join_kind, aliases, joined_from = _from_spec(chunks["FROM"])
    except ValueError as exc:
        return [{"code": "UNSUPPORTED_QUERY", "severity": "warning",
                 "message": str(exc), "details": {}}]

    predicate = chunks.get("WHERE", "")
    nullable = _nullable_columns(con, aliases, predicate) if predicate else []
    # A full null-safe predicate is deliberate. Do not suppress a warning just
    # because one clause is null-safe: another AND/OR clause can reject NULL.
    nullable = [(a, c) for a, c in nullable
                if not _whole_predicate_handles_null(predicate, a, c)]

    filter_sql = f" FILTER (WHERE ({predicate}))" if predicate else ""
    select_stats = [f"COUNT(*){filter_sql} AS joined_rows",
                    f"COUNT(DISTINCT {base_alias}.__review_rowid){filter_sql} AS matched_base_rows"]
    for index, (alias, column) in enumerate(nullable):
        select_stats.append(
            f"COUNT(*) FILTER (WHERE {alias}.{column} IS NULL AND ({predicate}) IS NOT TRUE) AS null_{index}")
    # The join appears once in this diagnostic statement; no full-result fetch.
    diagnostic = (f"WITH __review_base AS (SELECT *, ROW_NUMBER() OVER () AS __review_rowid "
                  f"FROM {base_table}) SELECT {', '.join(select_stats)} FROM {joined_from}")
    values = con.execute(diagnostic).fetchone()
    joined_rows, matched_base_rows = values[:2]
    # Apply a base-only WHERE to the baseline so the comparison uses the same
    # population (for example, West customers), not every customer in the table.
    other_aliases = [a for a in aliases if a != base_alias]
    base_predicate = predicate if predicate and not any(re.search(
        rf"\b{re.escape(a)}\.", predicate, re.I) for a in other_aliases) else ""
    try:
        base_rows = con.execute(
            f"SELECT COUNT(*) FROM {base_table} {base_alias}"
            + (f" WHERE {base_predicate}" if base_predicate else "")).fetchone()[0]
    except Exception:
        # An unqualified WHERE column can belong to the joined table.
        base_rows = con.execute(f"SELECT COUNT(*) FROM {base_table}").fetchone()[0]
    fanned_out_rows = joined_rows - matched_base_rows
    warnings = []
    if join_kind and (joined_rows != base_rows or fanned_out_rows > 0):
        warnings.append({"code": "JOIN_ROW_COUNT_CHANGE", "severity": "warning",
                         "message": "Join changes row multiplicity relative to the left table.",
                         "details": {"base_rows": base_rows, "joined_rows": joined_rows,
                                     "matched_base_rows": matched_base_rows,
                                     "fanned_out_rows": fanned_out_rows}})
    for index, (alias, column) in enumerate(nullable):
        warnings.append({"code": "NULL_FILTER", "severity": "warning",
                         "message": "WHERE can discard UNKNOWN for a nullable column.",
                         "details": {"column": f"{alias}.{column}",
                                     "dropped_null_rows": values[index + 2]}})
    if join_kind and fanned_out_rows > 0 and _AGGREGATE.search(chunks["SELECT"]):
        warnings.append({"code": "FANOUT_AGGREGATE", "severity": "warning",
                         "message": "Aggregate is computed after a fan-out; verify the intended grain.",
                         "details": {"fanned_out_rows": fanned_out_rows}})
    return warnings


*Your answer here.*

### Task 4.2: The one-page writeup (8 pts)

*Your answer here.*

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:*

*Prompt:*

*Reply:*

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.